# SOFIE Alpaka GPU Operator Tests
Tests for Tanh, Elu, and Softmax ONNX operators implemented using the alpaka heterogeneous library.

**Runtime:** Make sure to select **GPU (T4)** in Runtime > Change runtime type.

## GSoC 2026 - ML Inference on heterogeneous architectures using SOFIE
Exercise 3 & 4: Build the SOFIE alpaka standalone and test the implemented operators.

In [1]:
# Verify GPU is available
!nvidia-smi

Tue Mar  3 13:56:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq \
    cmake \
    libprotobuf-dev \
    protobuf-compiler \
    libgtest-dev \
    libopenblas-dev \
    git \
    wget \
    python3-pip
!pip install -q onnx onnxruntime

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../00-git_1%3a2.34.1-1ubuntu1.17_amd64.deb ...
Unpacking git (1:2.34.1-1ubuntu1.17) over (1:2.34.1-1ubuntu1.15) ...
Selecting previously unselected package googletest.
Preparing to unpack .../01-googletest_1.11.0-3_all.deb ...
Unpacking googletest (1.11.0-3) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../02-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libgtest-dev:amd64.
Preparing to unpack .../03-libgtest-dev_1.11.0-3_amd64.deb ...
Unpacking libgtest-dev:amd64 (1.11.0-3) ...
Selecting previously unselected package libjs-underscore.
Preparing to unpack .../04-libjs-underscore_1.13.2~dfsg-2_all.d

In [3]:
# Download and install ROOT pre-built binary (Ubuntu 22.04 / Python 3.10)
import os
!wget -q https://root.cern/download/root_v6.32.02.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
!tar -xzf root_v6.32.02.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
os.environ['ROOTSYS'] = '/content/root'
os.environ['PATH'] = '/content/root/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/content/root/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PYTHONPATH'] = '/content/root/lib:' + os.environ.get('PYTHONPATH', '')
!root --version

ROOT Version: 6.32.02
Built for linuxx8664gcc on Jun 18 2024, 04:33:34
From tags/v6-32-02@v6-32-02


In [12]:
# Clone the SOFIE fork with alpaka GPU operator implementations
!git clone -b gpu/alpaka https://github.com/harz05/SOFIE.git /content/SOFIE
!cd /content/SOFIE && git log --oneline -5

fatal: destination path '/content/SOFIE' already exists and is not an empty directory.
f7c86fe (HEAD -> gpu/alpaka, origin/gpu/alpaka) feat: implement Tanh, Elu, Softmax alpaka GPU operators with tests
671b4b0 fix: sigmoid operator gpu implementation and test
815a80c feat: test cases for leaky relu operator
59aeac4 feat: support for google tests for inference code with alpaka implementations
1979a11 feat: turn off emitting from ROOT files and skip tests with multiple output errors for now


In [24]:
!cd /content/SOFIE && git pull origin gpu/alpaka
!rm -rf /content/SOFIE/build && mkdir /content/SOFIE/build

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 620 bytes | 310.00 KiB/s, done.
From https://github.com/harz05/SOFIE
 * branch            gpu/alpaka -> FETCH_HEAD
   3427ff3..f892ebf  gpu/alpaka -> origin/gpu/alpaka
Updating 3427ff3..f892ebf
Fast-forward
 .../test/TestCustomModelsFromONNXForAlpakaCuda.cxx | 28 ++++++++--------------
 1 file changed, 10 insertions(+), 18 deletions(-)


In [25]:
# Verify our operator implementations are present
!grep -c 'TanhKernel\|Generate_GPU_Kernel_ALPAKA' /content/SOFIE/src/SOFIE_core/inc/SOFIE/ROperator_Tanh.hxx
!grep -c 'EluKernel\|Generate_GPU_Kernel_ALPAKA' /content/SOFIE/src/SOFIE_core/inc/SOFIE/ROperator_Elu.hxx
!grep -c 'SoftmaxKernel\|Generate_GPU_Kernel_ALPAKA' /content/SOFIE/src/SOFIE_core/inc/SOFIE/ROperator_Softmax.hxx
print('All 3 operators have alpaka GPU methods ✓')

3
3
3
All 3 operators have alpaka GPU methods ✓


In [26]:
# Build SOFIE standalone with alpaka CUDA tests enabled
import subprocess

!mkdir -p /content/SOFIE/build
result = subprocess.run([
    'cmake',
    '/content/SOFIE',
    '-DCMAKE_BUILD_TYPE=RelWithDebInfo',
    '-Dtesting=ON',
    '-DENABLE_ALPAKA_TESTS=ON',
    '-DALPAKA_BACKEND=cuda',
    f'-DCMAKE_INSTALL_PREFIX=/content/SOFIE/install',
    f'-DROOT_DIR=/content/root/cmake',
], cwd='/content/SOFIE/build', capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

-- The CXX compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Looking for sgemm_
-- Looking for sgemm_ - not found
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Looking for sgemm_
-- Looking for sgemm_ - found
-- Found BLAS: /usr/lib/x86_64-linux-gnu/libmkl_intel_lp64.so;/usr/lib/x86_64-linux-gnu/libmkl_intel_thread.so;/usr/lib/x86_64-linux-gnu/libmkl_core.so;/usr/local/lib/libiomp5.so;-lm;-ldl
-- Looking for Protobuf
-- Could NOT find Protobuf (missing: Protobuf_DIR)
-- Found Protobuf: /usr/lib/x86_64-linux-gnu/libprotobuf.so (found version "3.12.4")
-- Found Vdt: /content/root/include (found version "0.4")
-- Found GTest: /usr/lib/x86_64-linux-gnu/cmake/GTest/GTestConfig.cmake (found version "1.11.0")
-- Enabling

In [27]:
# Build (this will fetch alpaka and sofieBLAS via FetchContent)
!cd /content/SOFIE/build && cmake --build . --target install -j$(nproc) 2>&1 | tail -30

-- Installing: /content/SOFIE/install/include/alpaka/workdiv/Traits.hpp
-- Installing: /content/SOFIE/install/include/alpaka/workdiv/WorkDivMembers.hpp
-- Installing: /content/SOFIE/install/include/alpaka/workdiv/WorkDivHelpers.hpp
-- Installing: /content/SOFIE/install/include/alpaka/workdiv/WorkDivGenericSycl.hpp
-- Installing: /content/SOFIE/install/include/alpaka/workdiv/WorkDivUniformCudaHipBuiltIn.hpp
-- Up-to-date: /content/SOFIE/install/include/alpaka/offset
-- Installing: /content/SOFIE/install/include/alpaka/offset/Traits.hpp
-- Up-to-date: /content/SOFIE/install/include/alpaka/exec
-- Installing: /content/SOFIE/install/include/alpaka/exec/IndependentElements.hpp
-- Installing: /content/SOFIE/install/include/alpaka/exec/ElementIndex.hpp
-- Installing: /content/SOFIE/install/include/alpaka/exec/Once.hpp
-- Installing: /content/SOFIE/install/include/alpaka/exec/UniformElements.hpp
-- Installing: /content/SOFIE/install/lib/cmake/alpaka/addExecutable.cmake
-- Installing: /content/

In [28]:
# First emit (generate) the ONNX model headers for GPU
!cd /content/SOFIE/build && cmake --build . --target emitFromONNXAlpaka -j$(nproc) 2>&1 | tail -20
!cd /content/SOFIE/build && cmake --build . --target SofieCompileModels_ONNX_Alpaka -j$(nproc) 2>&1 | tail -20

[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/LSTMInitialBias.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/LSTMPeepholes.onnx



[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNBatchwise.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNBidirectional.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNBidirectionalBatchwise.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNDefaults.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNSeqLength.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/input_models/RNNSequence.onnx
[SKIP] Multiple outputs are not supported for /content/SOFIE/src/SOFIE_core/test/i

In [29]:
# Build the test executable
!cd /content/SOFIE/build && cmake --build . --target TestCustomModelsFromONNXForAlpakaCuda -j$(nproc) 2>&1 | tail -30

            instantiation of "auto alpaka::rand::engine::PhiloxStateless<TParams>::singleRound(const alpaka::rand::engine::PhiloxStateless<TParams>::Counter &, const alpaka::rand::engine::PhiloxStateless<TParams>::Key &) [with TParams=alpaka::rand::Philox4x32x10::EngineParams]" at line 101
            instantiation of "auto alpaka::rand::engine::PhiloxStateless<TParams>::nRounds(const alpaka::rand::engine::PhiloxStateless<TParams>::Counter &, const alpaka::rand::engine::PhiloxStateless<TParams>::Key &)->alpaka::rand::engine::PhiloxStateless<TParams>::Counter [with TParams=alpaka::rand::Philox4x32x10::EngineParams]" at line 65 of /content/SOFIE/build/_deps/alpaka-src/include/alpaka/rand/Philox/PhiloxVector.hpp

/content/SOFIE/build/_deps/alpaka-src/include/alpaka/rand/Philox/PhiloxStateless.hpp(103): warning #20013-D: calling a constexpr __host__ function("numRounds") from a __host__ __device__ function("nRounds") is not allowed. The experimental flag '--expt-relaxed-constexpr' can be u

In [32]:
!cd /content/SOFIE/build/src/SOFIE_core/test && ./TestCustomModelsFromONNXForAlpakaCuda --gtest_color=yes 2>&1

Running main() from ./googletest/src/gtest_main.cc
[==========] Running 8 tests from 1 test suite.
[----------] Global test environment set-up.
[----------] 8 tests from SofieAlpakaTest
[ RUN      ] SofieAlpakaTest.Linear16
[       OK ] SofieAlpakaTest.Linear16 (443 ms)
[ RUN      ] SofieAlpakaTest.Linear32
[       OK ] SofieAlpakaTest.Linear32 (48 ms)
[ RUN      ] SofieAlpakaTest.Linear64
[       OK ] SofieAlpakaTest.Linear64 (18 ms)
[ RUN      ] SofieAlpakaTest.LinearWithLeakyRelu
[       OK ] SofieAlpakaTest.LinearWithLeakyRelu (417 ms)
[ RUN      ] SofieAlpakaTest.LinearWithSigmoid
[       OK ] SofieAlpakaTest.LinearWithSigmoid (2 ms)
[ RUN      ] SofieAlpakaTest.Tanh
[       OK ] SofieAlpakaTest.Tanh (1 ms)
[ RUN      ] SofieAlpakaTest.Elu
[       OK ] SofieAlpakaTest.Elu (1 ms)
[ RUN      ] SofieAlpakaTest.Softmax1d
[       OK ] SofieAlpakaTest.Softmax1d (1 ms)
[----------] 8 tests from SofieAlpakaTest (935 ms total)

[----------] Global test environment tear-down
[==========] 8 